# Exact Adversarial Attacks on an MNIST CNN

This notebook trains a compact convolutional classifier on MNIST and converts it into an exact NCET mixed-integer formulation. For one correctly classified test sample, it solves:

1. a **targeted attack** that maximizes the selected target class's margin over every other class; and
2. a **worst-possible untargeted attack** that maximizes the largest incorrect-class margin over the true class.

The perturbation is constrained by the elementwise box

$$\max(0,x_0-\varepsilon)\leq x\leq\min(1,x_0+\varepsilon),$$

which is the intersection of the valid pixel range and an $L_\infty$ ball. The attack acts on the original $28\times28$ pixels. The CNN downsamples internally with an NCET-supported `AvgPool2d` layer to keep the exact MILPs small.

Run this notebook from the NCET repository after installing the example dependencies with `pip install -e '.[examples]'`.

In [ ]:
from __future__ import annotations

from pathlib import Path
from urllib.request import urlretrieve

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from ncet import Bounds, form_milp

SEED = 7
BATCH_SIZE = 256
EPOCHS = 15
LEARNING_RATE = 3e-3
EPSILON = 0.30

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training device:", device)

## 1. Download and load MNIST

The archive is cached under `~/.cache/ncet`. Images remain as `uint8` in memory and are converted to floating-point tensors only when a batch is requested.

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
cache_path = Path.home() / ".cache" / "ncet" / "mnist.npz"
cache_path.parent.mkdir(parents=True, exist_ok=True)
if not cache_path.exists():
    print("downloading MNIST...")
    urlretrieve(MNIST_URL, cache_path)

with np.load(cache_path) as data:
    train_images = data["x_train"]
    train_labels = data["y_train"]
    test_images = data["x_test"]
    test_labels = data["y_test"]

print(train_images.shape, test_images.shape)

In [ ]:
class MNISTArrayDataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray) -> None:
        self.images = torch.from_numpy(images)
        self.labels = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        image = self.images[index].float().unsqueeze(0) / 255.0
        return image, self.labels[index]


train_dataset = MNISTArrayDataset(train_images, train_labels)
test_dataset = MNISTArrayDataset(test_images, test_labels)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

## 2. Train a compact supported CNN

The model contains only operators in NCET's current exact boundary:

```text
Input(1, 28, 28)
  -> AvgPool2d(4, 4)            # (1, 7, 7)
  -> Conv2d(1, 4, 3, stride=2) # (4, 4, 4)
  -> ReLU -> Flatten
  -> Linear(64, 24) -> ReLU
  -> Linear(24, 10)             # logits
```

The two ReLU tensors contain only $64+24=88$ elements before NCET removes stable activations in reduced mode.

In [ ]:
class TinyMNISTCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.pool = nn.AvgPool2d(kernel_size=4, stride=4)
        self.conv = nn.Conv2d(1, 4, kernel_size=3, stride=2, padding=1)
        self.relu1 = nn.ReLU()
        self.flatten = nn.Flatten(start_dim=1)
        self.hidden = nn.Linear(4 * 4 * 4, 24)
        self.relu2 = nn.ReLU()
        self.output = nn.Linear(24, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(x)
        x = self.relu1(self.conv(x))
        x = self.flatten(x)
        x = self.relu2(self.hidden(x))
        return self.output(x)


model = TinyMNISTCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_function = nn.CrossEntropyLoss()
model

In [ ]:
def classification_accuracy(
    classifier: nn.Module, loader: DataLoader, device: torch.device
) -> float:
    classifier.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            predictions = classifier(images.to(device)).argmax(dim=1).cpu()
            correct += int((predictions == labels).sum())
            total += len(labels)
    return correct / total


for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_function(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.detach()) * len(labels)

    accuracy = classification_accuracy(model, test_loader, device)
    print(
        f"epoch {epoch:2d}: loss={running_loss / len(train_dataset):.4f}, "
        f"test_accuracy={accuracy:.3%}"
    )

model = model.cpu().eval()

## 3. Select one correctly classified sample

The notebook randomly selects one correctly classified image from the first 512 test samples. The fixed `SEED` makes the choice reproducible.

In [ ]:
candidate_images = torch.stack([test_dataset[i][0] for i in range(512)])
candidate_labels = torch.tensor([int(test_dataset[i][1]) for i in range(512)])

with torch.no_grad():
    candidate_predictions = model(candidate_images).argmax(dim=1)
correct_indices = torch.where(candidate_predictions == candidate_labels)[0]

if not len(correct_indices):
    raise RuntimeError("no correctly classified candidate was found. Please train the model longer.")

sample_index = int(np.random.default_rng(SEED).choice(correct_indices.numpy()))
sample, label_tensor = test_dataset[sample_index]
true_class = int(label_tensor)
with torch.no_grad():
    clean_logits = model(sample.unsqueeze(0)).squeeze(0).numpy()
clean_prediction = int(clean_logits.argmax())
print(
    f"sample={sample_index}, true={true_class}, prediction={clean_prediction}, "
    f"clean_margin={clean_logits[true_class] - np.max(np.delete(clean_logits, true_class)):.4f}"
)

In [ ]:
plt.imshow(sample.squeeze().numpy(), cmap="gray", vmin=0.0, vmax=1.0)
plt.title(f"Clean prediction: {clean_prediction}")
plt.axis("off")
plt.show()

## 4. Build one exact NCET formulation

NCET receives per-sample bounds of shape `(1, 28, 28)`; no batch dimension is included. The same formulation is reused with different linear objectives. `reduced` mode remains exact but omits binary variables for ReLU elements proven stable by interval bound propagation.

In [ ]:
# Define the input Linf box
sample_array = sample.numpy()
lower = np.clip(sample_array - EPSILON, 0.0, 1.0).astype(np.float32)
upper = np.clip(sample_array + EPSILON, 0.0, 1.0).astype(np.float32)

# Convert the NN model into MIL set of constraints using NCET
encoding = form_milp(
    model, Bounds(lower=lower, upper=upper), relu_binary_mode="reduced"
) # alternatively, input bounds can be provided as a tuple (lower, upper), list of tuples for multiple inputs, and dictionary {x: (lower, upper)}

# Extract the input and output cvxpy expressions
input_expression = encoding.inputs["x"]
logit_expression = encoding.outputs[0]

MIP_SOLVER = cp.SCIPY

print("solver:", MIP_SOLVER)
print(encoding.stats)

## 5. Exact targeted attack

For target class $t$, solve

$$\max_{x,\gamma}\;\gamma$$

subject to the NCET constraints, the input box, and

$$f_t(x)-f_k(x) \geq \gamma,\qquad k\neq t.$$

Thus $\gamma^*=\max_x\min_{k\neq t}(f_t(x)-f_k(x))$. If $\gamma^*>0$, the target logit is strictly larger than every other logit and the targeted attack succeeds. The target is the second-highest-scoring class on the clean, correctly classified sample.

In [ ]:
target_class = int(np.argsort(clean_logits)[-2])

# Define new variable gamma
gamma = cp.Variable(name=f"target_{target_class}_dominance")

# Add new constraints with respect to the encoding.outputs[0] expression
dominance_constraints = [
    gamma <= logit_expression[target_class] - logit_expression[other]
    for other in range(10)
    if other != target_class
]

# Formulate the targeted attack problem by adding encoding.constraints and dominance_constraints
targeted_problem = cp.Problem(
    cp.Maximize(gamma), [*encoding.constraints, *dominance_constraints]
)

# Solve optimization
targeted_dominance = targeted_problem.solve(solver=MIP_SOLVER)
if targeted_problem.status not in {cp.OPTIMAL, cp.OPTIMAL_INACCURATE}:
    raise RuntimeError(f"targeted attack failed: {targeted_problem.status}")

# Extract the optimized input and output values
targeted_image = np.asarray(input_expression.value).copy()
targeted_logits = np.asarray(logit_expression.value).copy()

# Get the predicted class from the optimized logits
targeted_prediction = int(targeted_logits.argmax())

# Print the results
print(
    f"target={target_class}, dominance={targeted_dominance:.6f}, "
    f"prediction={targeted_prediction}"
)
print("targeted attack successful:", targeted_dominance > 1e-6)

# Plot the results
plt.imshow(targeted_image.squeeze(), cmap="gray", vmin=0.0, vmax=1.0)
plt.title(f"Targeted attack: {target_class} -> {targeted_prediction}")
plt.axis("off")
plt.show()

## 6. Verify the optimized points with PyTorch

The final check evaluates both optimizer-produced images directly with the trained PyTorch model, compares those logits with the NCET variables, and verifies the $L_\infty$ and pixel-range constraints.

In [ ]:
with torch.no_grad():
    tensor = torch.from_numpy(targeted_image).to(dtype=sample.dtype).unsqueeze(0)
    torch_logits = model(tensor).squeeze(0).numpy()
np.testing.assert_allclose(targeted_logits, torch_logits, atol=2e-4, rtol=2e-4)
linf = float(np.max(np.abs(targeted_image - sample_array)))
assert linf <= EPSILON + 1e-5
assert targeted_image.min() >= -1e-5 and targeted_image.max() <= 1.0 + 1e-5
print(f"prediction={torch_logits.argmax()}, L_inf={linf:.6f}")

## Interpretation

These are global optima for the trained network over the supplied pixel box, subject to mixed-integer solver tolerances. 

Changing the model, sample, target, or `EPSILON` requires rebuilding the NCET formulation because the propagated bounds and big-M coefficients depend on the input box.